# Genesis Terminal – Patch Finale Colab

Allinea una repo caricata su Colab al pacchetto finale Analysis Studio e verifica i test minimi.

In [ ]:
# 1. Environment Setup & DATA_PATH
import os
import sys
import subprocess
import platform
import shutil
import zipfile
from pathlib import Path

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    %pip -q install pandas numpy yfinance plotly loguru pyarrow typer rich streamlit pytest tabulate nbformat
else:
    print("Running locally:", platform.platform())

DEFAULT_LOCAL_DATA_PATH = Path("/Users/itsgennymac/Library/CloudStorage/GoogleDrive-sfn.gns@gmail.com/Il mio Drive/Database Finanziario")
DEFAULT_COLAB_DATA_PATH = Path("/content/drive/MyDrive/Database Finanziario")
DATA_PATH = Path(os.environ.get(
    "DATA_PATH",
    str(DEFAULT_COLAB_DATA_PATH if IN_COLAB else DEFAULT_LOCAL_DATA_PATH),
)).expanduser()
try:
    DATA_PATH.mkdir(parents=True, exist_ok=True)
except Exception as exc:
    print(f"[WARNING] DATA_PATH could not be created: {DATA_PATH} ({exc})")
os.environ["DATA_PATH"] = str(DATA_PATH)
DB_BASE = DATA_PATH

print("IN_COLAB:", IN_COLAB)
print("DATA_PATH:", DATA_PATH)


In [ ]:
# 2. Configuration & Paths
from pathlib import Path

REPO_DIR = Path("/content/ml-trading-thesis-bot") if IN_COLAB else Path.cwd().resolve()
DB_BASE = DATA_PATH
for relative_path in ['analysis_outputs', 'notebook_exports/html', 'notebook_exports/markdown', 'notebook_exports/csv', 'notebook_exports/charts', 'logs']:
    (DB_BASE / relative_path).mkdir(parents=True, exist_ok=True)

print("REPO_DIR:", REPO_DIR)
print("DB_BASE:", DB_BASE)


In [ ]:
# 3. Quick repository check
required = [
    "src/reporting/report_generator.py",
    "src/cli/analysis_commands.py",
    "scripts/weekly_update.py",
    "tests/test_analysis_commands.py",
    "tests/test_weekly_update.py",
]
for rel in required:
    print(rel, "OK" if (REPO_DIR / rel).exists() else "MISSING")


In [ ]:
# 4. Run minimal tests
if (REPO_DIR / "tests").exists():
    result = subprocess.run([sys.executable, "-m", "pytest", "tests/test_analysis_commands.py", "tests/test_weekly_update.py", "-q"], cwd=str(REPO_DIR), text=True, capture_output=True)
    print(result.stdout)
    print(result.stderr)
    assert result.returncode == 0
else:
    print("Tests folder missing: upload/apply final package first.")


In [ ]:
# 5. Run a quick smoke command
result = subprocess.run([sys.executable, "-m", "src.cli.analysis_commands", "--help"], cwd=str(REPO_DIR), text=True, capture_output=True)
print(result.stdout)
assert result.returncode == 0
